# Assignment

- **i.** session from the LOBSTER pair (Section 1.2) and reconciliation with the orderbook file;
- **ii.** $\mathrm{OFI}_{t,w}$ (Definition 1.4.2) and $I^1_t$ (1.1.2);
- **iii.** regression of $\Delta P^m_{t,w}$ on $\mathrm{OFI}_{t,w}$ and comparison with the slope $\tau/(2\bar S)$ of (1.4.3);
- **iv.** sign and strength of the relation between $\mathrm{OFI}_{t,w}$ and $P^m_{t+h}-P^m_t$ over a grid of $(w,h)$;
- **v.** the same with $I^1_t$, and the comparison.

**Data.** AMZN, 21 June 2012, level 10, from 10:00 to 13:00, away from the open and the close (lecture 01, §8). Prices are integer ticks and the price unit is derived from the tick size (`lobster.price_unit`). The orderbook file is aligned to the message file by position only (Section 1.2.5).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from unito26.lob import lobster
from unito26.lob.messages import GridDepth, ReportedDepth, TickGrid
from unito26.lob.statistics import SessionStatistics
from unito26.lob.visualization import use_template

use_template()

TICKER = "AMZN"
DAY = "2012-06-21"
GRID = TickGrid(0.01)
DEPTH = ReportedDepth(10)
UNIT = lobster.price_unit(GRID)
WINDOW = lobster.TradingWindow(36000.0, 46800.0)  # 10:00 to 13:00, seconds after midnight
SPEC = SessionStatistics((GridDepth(1),), (), ())

DATA = Path("..") / "data" / "lobster"


def message_file(ticker):
    return DATA / f"{ticker}_{DAY}_34200000_57600000_message_10.csv"


FILES = lobster.LobsterFiles.parse(message_file(TICKER))

print(f"{TICKER}, {DAY}, level {DEPTH}")
print(f"window: {WINDOW.opens:.0f} s to {WINDOW.closes:.0f} s after midnight")
print(f"price unit: {UNIT} file units per tick")

## i. The session from the pair of files

The file has one row per execution of a resting order, while a session has one row per order. `coarsened` groups the executions of one market order and keeps its last row (lecture 02, §7-8). Volume, traded value and VWAP are unchanged; the number of rows is not, and by Remark 1.2.6 the contribution $e_n$ of the coarse row equals the sum over the fills unless the order walks the book.

Point i. works on the raw pair and points ii.–v. on the coarse session, where one aggressive order is one event, as in Proposition 1.4.9.

In [ ]:
from unito26.lob.lobster_session import LobsterMarketSession

raw = LobsterMarketSession.from_files(FILES, SPEC, GRID, WINDOW)
session = raw.coarsened(True)

print(f"raw pair, rows         : {len(raw.messages):,}")
print(f"coarse session, rows   : {len(session.lobster_book):,}")

**Result.** The raw pair has 125,276 rows and the coarse session 123,324. The difference of 1,952 rows is the effect of the coarsening: the visible fills of one market order are merged into its last row, and the hidden executions, which move no visible level, are dropped or absorbed. The two parts are not counted separately here.

### The fold of the message file

Section 1.2.2 writes the book as the accumulator of a fold, $\mathcal B_k = F(\mathcal B_{k-1}, m_k)$ (1.2.1), with $F$ given by Algorithm 1.2.1. The package has the accumulator (`AggregateBook`) but no map from a LOBSTER row to a message (lecture 02, §2), so the map is written inside the loop:

- type 1 (submission) → `limit_order` at the stated price and size;
- types 2, 3 and 4 (partial cancellation, deletion, visible execution) → `withdrawal` of the stated size at the stated price, on the side the row carries (Remark 1.2.5);
- types 5 and 7 (hidden execution, halt) → nothing, they move no visible level.

The loop is run twice:

- **seeded once**: the book starts from the first row of the orderbook file and folds all the following messages;
- **reseeded at every message**: the book is rebuilt from row $k-1$ of the file and receives only message $k$, to be compared with row $k$.

The book is not strict, so a withdrawal larger than the size resting is cut to what rests.

In [ ]:
from unito26.lob.messages import limit_order, withdrawal
from unito26.lob.orderbook import AggregateBook

messages = raw.messages
n = len(messages)
times = messages["Time"].to_numpy()
types = messages["Type"].to_numpy()
sizes = messages["Size"].to_numpy()
prices = messages["Price"].to_numpy()
directions = messages["Direction"].to_numpy()
recorded = raw.book.to_numpy()

SUBMISSION = lobster.LobsterEvent.SUBMISSION
LEAVES_THE_BOOK = (lobster.LobsterEvent.PARTIAL_CANCELLATION,
                   lobster.LobsterEvent.DELETION,
                   lobster.LobsterEvent.EXECUTION_VISIBLE)

folds = {}
for mode in ("seeded once", "reseeded at every message"):
    book = AggregateBook.from_lobster_row(raw.book.iloc[0], UNIT, DEPTH)
    folded = recorded.copy()
    for k in range(1, n):
        if mode == "reseeded at every message":
            book = AggregateBook.from_lobster_row(raw.book.iloc[k - 1], UNIT, DEPTH)
        kind = int(types[k])
        if kind == SUBMISSION:
            tick = int(prices[k]) // UNIT
            book.apply(limit_order(float(times[k]), int(sizes[k]), tick, int(directions[k])),
                       record=False)
        elif kind in LEAVES_THE_BOOK:
            tick = int(prices[k]) // UNIT
            book.apply(withdrawal(float(times[k]), int(sizes[k]), tick, int(directions[k])),
                       record=False)
        folded[k] = book.to_lobster_row(UNIT, DEPTH)
    folds[mode] = folded

print("messages folded         :", n - 1)
print("  as submissions        :", int((types[1:] == SUBMISSION).sum()))
print("  as withdrawals        :", int(np.isin(types[1:], LEAVES_THE_BOOK).sum()))
print("  ignored (hidden/halt) :", int((~np.isin(types[1:], (SUBMISSION, *LEAVES_THE_BOOK))).sum()))

**Result.** The fold applies 125,275 messages: 61,677 submissions, 62,668 withdrawals and 930 hidden executions, which are ignored. The first message is absorbed by the seed.

### Reconciliation with the orderbook file

The orderbook file is the exchange's own fold of the same messages, so comparing the two state by state locates a mistake of the reconstruction (Remark 1.2.3, lecture 02, §5). Row $k$ of the file is the configuration after message $k$ (Section 1.2.5).

The file reports 10 occupied levels, and the message file reports only the events inside that window. An order posted beyond the 10th level has no message and appears in the file, with none, when the window moves outward (Remark 1.2.4). The cell checks:

1. how many messages fall beyond the 10th level of the state before them;
2. for each fold, the share of rows equal to the file on all 10 levels, on every price inside the window of the previous row, and on the touch.

In [ ]:
rec = recorded.reshape(n, DEPTH, 4)  # ask price, ask size, bid price, bid size
last_ask, last_bid = rec[:, -1, 0], rec[:, -1, 2]
before = np.maximum(np.arange(n) - 1, 0)

# 1. messages beyond the 10th level of the state before
beyond = np.where(directions == -1, prices > last_ask[before], prices < last_bid[before])
beyond[0] = False
in_book = np.isin(types, (SUBMISSION, *LEAVES_THE_BOOK))
census = pd.DataFrame({"type": types[in_book], "beyond": beyond[in_book]}
                      ).groupby("type")["beyond"].agg(messages="size", beyond_10th_level="sum")
census.index = [lobster.LobsterEvent(int(t)).name for t in census.index]
print("messages beyond the 10th level of the state before:")
print(census.to_string())

# 2. the two folds against the file
summary = {}
for mode, folded in folds.items():
    rows = folded.reshape(n, DEPTH, 4)
    keep = np.concatenate(
        [np.repeat((rows[..., 0] <= last_ask[before][:, None])[..., None], 2, axis=2),
         np.repeat((rows[..., 2] >= last_bid[before][:, None])[..., None], 2, axis=2)], axis=2)
    keep_rec = np.concatenate(
        [np.repeat((rec[..., 0] <= last_ask[before][:, None])[..., None], 2, axis=2),
         np.repeat((rec[..., 2] >= last_bid[before][:, None])[..., None], 2, axis=2)], axis=2)
    in_window = (np.where(keep, rows, 0) == np.where(keep_rec, rec, 0)).all(axis=(1, 2))
    touch = (rows[:, 0] == rec[:, 0]).all(axis=1)
    summary[mode] = {
        "rows identical on all 10 levels": f"{(rows == rec).all(axis=(1, 2)).mean():.1%}",
        "rows identical inside the previous window": f"{in_window.mean():.1%}",
        "rows with the touch identical": f"{touch.mean():.1%}",
    }

print()
print(pd.DataFrame(summary).to_string())

**Result.** No message falls beyond the 10th level of the state before it (0 out of 124,346).

- **Reseeded at every message**, the fold equals the file on the touch and on every price inside the previous window in 100% of the rows. The translation of the messages is consistent with the orderbook file. The 21.5% of rows that differ on the 10 levels differ only beyond the 10th level of the previous row, the levels that the window uncovers (Remark 1.2.4).
- **Seeded once**, the fold drifts: the touch is identical in 25.5% of the rows. The orders beyond the window are never announced by a message, so the fold cannot know them.

The discrepancies of the reconstruction are therefore the orders beyond the 10th level.

## ii. $\mathrm{OFI}_{t,w}$ and $I^1_t$

### The order flow imbalance

Definition 1.4.2 gives the contribution of the $n$-th event (1.4.1), which reads the touch before and after the event, and the imbalance over a window as the sum of the contributions (1.4.2):

$$\mathrm{OFI}_{t,w} = \sum_{n:\,t-w<T_n\le t} e_n .$$

We use the coarse session, where one row is one order (Proposition 1.4.9). The session records $e_n$ in `OrderFlowContribution` and computes $\mathrm{OFI}$ for any window with `order_flow_imbalance` (lecture 07, §2-3).

The window is $(t-w,t]$, closed on the right at the current row. $e_n$ is undefined on the first row, so every window containing it is NaN (lecture 07, §2).

In [ ]:
from unito26.lob.messages import Window

e_n = session.stats["OrderFlowContribution"].to_numpy()
ofi = session.order_flow_imbalance(Window(1.0)).to_numpy()

print("rows                          :", len(e_n))
print("e_n undefined (NaN)           :", int(np.isnan(e_n).sum()))
print("OFI, w = 1 s: undefined rows  :", int(np.isnan(ofi).sum()))

**Result.** The only undefined $e_n$ is the first row. The 83 undefined values of $\mathrm{OFI}$ at $w=1$ s are the rows whose window reaches back to it.

### The queue imbalance

$I^1_t$ is Definition 1.1.6, equation (1.1.2) at $n=1$:

$$I^1_t = \frac{S^{b,1}_t - S^{a,1}_t}{S^{b,1}_t + S^{a,1}_t},$$

positive when the book is bid-heavy. The session records it in `QueueImbalance1`. At $n=1$ the grid indexing of the notes and the occupied-level indexing of the file coincide, since the first level is the touch on both (lecture 01, §8). The cell reads the column and prints the number of undefined values.

In [ ]:
i1 = session.stats["QueueImbalance1"].to_numpy()

print("I^1 undefined (NaN) :", int(np.isnan(i1).sum()))

**Result.** $I^1_t$ has no undefined value, so every row enters point v.

## iii. The contemporaneous regression

### Slope and implied depth

Proposition 1.4.3 gives the mid-price change over a window that has elapsed:

$$\Delta P^m_{t,w} = \frac{\tau}{2\bar S}\,\mathrm{OFI}_{t,w} + \varepsilon_{t,w}, \qquad |\varepsilon_{t,w}|\le\tau \qquad (1.4.3).$$

We regress $\Delta P^m_{t,w}$ on $\mathrm{OFI}_{t,w}$ through the origin, as (1.4.3) has no intercept, for several windows $w$. The mid-price is in ticks, so $\tau=1$ and the slope is in ticks per share; in dollars it is multiplied by $\tau=0.01$ (Section 1.4.1). If (1.4.3) held with a constant depth the slope would be $1/(2\bar S)$, so the **implied depth** is

$$\bar S_{\text{implied}} = \frac{1}{2\,\text{slope}}.$$

Rows are sampled at events and the windows overlap, so we use the Newey-West standard error (`hac_variance`). The lag is four times the mean number of rows in the overlap, which here is $w$ (the docstring of `unito26.lob.estimation` sets it by the forecast horizon). The rows are those where $\Delta P^m_{t,w}$ and $\mathrm{OFI}_{t,w}$ are both defined (`backward_change`, `backward_sum`; lecture 07, §3-4). The cell prints, for each $w$, the slope, its standard error and the implied depth.

In [ ]:
from unito26.lob.estimation import hac_variance, ols
from unito26.lob.imbalance_regression import (
    aligned_mid_price, aligned_order_flow, backward_change, backward_sum,
)

mid = aligned_mid_price(session)
flow = aligned_order_flow(session)
rate = len(mid) / (mid.times[-1] - mid.times[0])  # rows per second

table = []
for w in (1.0, 5.0, 10.0, 30.0, 60.0):
    change = backward_change(mid, Window(w))
    imbalance = backward_sum(flow, Window(w))
    ok = np.isfinite(change) & np.isfinite(imbalance)
    fit = ols(imbalance[ok, None], change[ok])
    slope = float(fit.coefficients[0])
    lag = int(4 * w * rate)
    se = float(np.sqrt(hac_variance(fit, max(lag, 1))[0, 0]))
    table.append({
        "w (s)": w, "rows": int(ok.sum()), "lag": lag,
        "slope": slope, "s.e.": se,
        "implied depth": 1 / (2 * slope) if slope > 0 else np.nan,
    })

print("regression of the mid-price change (ticks) on OFI (shares), through the origin")
print(pd.DataFrame(table).set_index("w (s)").round(
    {"slope": 6, "s.e.": 6, "implied depth": 0}).to_string())

**Result.** The slope is positive at every window and more than two standard errors from zero (4.4 at $w=1$ s, between 2.7 and 2.9 at the others). It grows from 0.00104 ticks per share at $w=1$ s to 0.00325 at $w=60$ s, so the implied depth falls from 479 to 154 shares. In dollars the slope at $w=1$ s is about $1.0\cdot10^{-5}$ per share.

(1.4.3) with a constant $\bar S$ would give the same slope at every window, so the data do not give one depth. The growth is in line with Remark 1.4.5, which bounds the residual by one tick whatever $w$ while the flow term grows with it, but the two-standard-error bands of $w=1$ s and $w=60$ s overlap, so it is not tested.

### Implied depth against the observed depth

A recorded book has no $\bar S$. As a reference we take the size resting at the touch, $S_{\text{obs}}$, the mean over the rows of $\tfrac12\,(S^{b,1}_t+S^{a,1}_t)$. The theoretical slope of (1.4.3) at that depth is $1/(2S_{\text{obs}})$ (with $\tau=1$), and the ratio of the estimated slope to it says how far the data are from (1.4.3) with $\bar S=S_{\text{obs}}$. The cell prints $S_{\text{obs}}$ and, for each $w$, the slope, the implied depth, the theoretical slope and the two ratios.

In [ ]:
book_s = session.lobster_book
bid_q, ask_q = book_s["BidSize1"].to_numpy(), book_s["AskSize1"].to_numpy()
s_obs = ((bid_q + ask_q) / 2).mean()
theory = 1 / (2 * s_obs)

print(f"touch depth (S^b + S^a)/2, shares: mean {s_obs:.1f}")
print()

result = pd.DataFrame(table).set_index("w (s)")[["slope", "implied depth"]]
result["theory 1/(2 S_obs)"] = theory
result["slope / theory"] = result["slope"] / theory
result["implied / observed depth"] = result["implied depth"] / s_obs
print(result.round({"slope": 6, "implied depth": 0, "theory 1/(2 S_obs)": 6,
                    "slope / theory": 2, "implied / observed depth": 2}).to_string())

**Result.** The size at the touch averages 194.7 shares. The estimated slope is 0.41 of the theoretical one at $w=1$ s and 1.27 at $w=60$ s, crossing 1 near $w=30$ s; the implied depth goes from 2.5 times the observed one to 0.8 times.

(1.4.3) with a constant depth predicts a ratio of 1 at every window, so the crossing is not a confirmation.

Two caveats on the reference. $S_{\text{obs}}$ is a convention: in Assumption 1.4.1 $\bar S$ is the size of every queue behind the touch, not the size at the touch, and a recorded book has no such constant. And the assumption itself, a full queue on every tick behind the touch, is far from AMZN, a small-tick instrument whose levels are spread over many grid positions (lecture 01, §8).

### What (1.4.4) accounts for

Equation (1.4.4) gives the residual of (1.4.3) in terms of the touch sizes at the two ends of the window:

$$\varepsilon_{t,w} = -\frac{\tau}{2\bar S}\,Q_{t,w}, \qquad Q_{t,w} = \big(S^{b,1}_t - S^{b,1}_{t-w}\big) - \big(S^{a,1}_t - S^{a,1}_{t-w}\big).$$

Together with (1.4.3) this gives $\Delta P^m_{t,w} = c\,(\mathrm{OFI}_{t,w} - Q_{t,w})$ with $c=\tau/(2\bar S)$. We regress $\Delta P^m_{t,w}$ on $\mathrm{OFI}_{t,w}$ and $Q_{t,w}$ together, through the origin, with coefficients $b_1$ and $b_2$; the identity predicts $b_1=c$ and $b_2=-c$.

The notes do not say how to split the gap between the slope and $c$; the split used here is least squares algebra. With $\hat b$ the slope of the previous regression and $\delta$ the slope of $Q$ on $\mathrm{OFI}$, $\hat b = b_1 + b_2\,\delta$, so $b_2\delta$ is the part of $\hat b$ that comes from leaving $Q$ out. Against $c = 1/(2S_{\text{obs}})$ we report the gap $\hat b - c$, the gap left once $Q$ is in, $b_1 - c$, and the share $(\hat b - b_1)/(\hat b - c)$ that (1.4.4) accounts for. $Q$ is built on the same rows as $\Delta P^m$, with the base at the row before the window opened (`backward_change`; lecture 07, §3-4).

In [ ]:
from unito26.lob.imbalance_regression import AlignedSeries

queues = AlignedSeries(times=mid.times, values=(bid_q - ask_q).astype(float), segments=mid.segments)

rows_out = []
for entry in table:
    w = entry["w (s)"]
    change = backward_change(mid, Window(w))
    imbalance = backward_sum(flow, Window(w))
    q = backward_change(queues, Window(w))
    ok = np.isfinite(change) & np.isfinite(imbalance) & np.isfinite(q)
    b1, b2 = ols(np.column_stack([imbalance[ok], q[ok]]), change[ok]).coefficients
    delta = ols(imbalance[ok, None], q[ok]).coefficients[0]
    slope = entry["slope"]
    rows_out.append({
        "w (s)": w, "slope": slope, "b1": b1, "b2": b2, "delta": delta,
        "slope - theory": slope - theory, "b1 - theory": b1 - theory,
        "share by (1.4.4)": (slope - b1) / (slope - theory),
    })

print(f"theory 1/(2 S_obs) = {theory:.6f}")
print()
print(pd.DataFrame(rows_out).set_index("w (s)").round(6).to_string())

**Result.** $b_1$ is positive and $b_2$ negative at every window, as the identity predicts, but $|b_2|<b_1$ (0.0020 against 0.0031 at $w=5$ s), so $b_2=-b_1$ does not hold exactly. As a check, $b_1+b_2\delta$ reproduces the slope of the previous regression (at $w=1$ s, $0.004272-0.003580\cdot0.902=0.001044$).

The slope is below $b_1$ at every window, so leaving $Q$ out attenuates it, in the direction of (1.4.4). $\delta$ falls from 0.90 at $w=1$ s to 0.13 at $w=60$ s: at short windows much of the flow goes into the queues (Remark 1.4.5).

The share is readable at $w=1$, $5$ and $10$ s, where $\hat b-c$ is $-0.0015$, $-0.0010$ and $-0.0007$. It is 2.1, 1.6 and 1.05: at 1 and 5 s adding $Q$ more than closes the gap ($b_1-c=+0.0017$ and $+0.0006$), at 10 s it closes it ($b_1-c=+0.00003$). At $w=30$ s the gap is almost zero (ratio 5.3) and at $w=60$ s the slope is above the theory (negative ratio), so there the share says nothing. There are no standard errors for $b_1$ and $b_2$.

## iv. Sign and strength of $\mathrm{OFI}_{t,w}$ against the future mid-price change

For each pair we take $\mathrm{OFI}_{t,w}$ and $P^m_{t+h}-P^m_t$ on the coarse session, on the rows where both are defined (`backward_sum`, `forward_change`), and compute:

- **sign**: $\mathrm{Cov}\big(\operatorname{sign}\mathrm{OFI}_{t,w},\,\Delta P^m\big)$ in ticks (`signed_covariance`), positive when buying pressure is followed by a rising mid. It grows with $h$ because $\Delta P^m$ does, so it is not compared across horizons;
- **strength**: Kendall's $\tau_b$ (`kendall_tau_b`), free of the scale of $\mathrm{OFI}$ and of $h$, and comparable across cells;
- **interval** for the sign statistic: 95% percentile interval from the block bootstrap over blocks of time (`bootstrap_means`, 300 replicates). The windows overlap over $w+h$ seconds, so the block duration is the larger of the Politis-White length (`politis_white_block_length`, converted to seconds) and $w+h$.

A cell is **readable** when the interval excludes zero. The cell prints the table of the 25 cells, with the column `read` equal to `+` or `-` if the interval excludes zero, `.` if it includes it.

**Row set.** `forward_change` is defined on a row only if some later row falls in $(t, t+h]$, so the number of rows depends on $h$: rows followed by no event within the horizon are dropped (2.0% of the rows at $h=1$ s, a handful at $h\ge5$ s), and in the last $h$ seconds of the window the horizon is truncated at the last row (about 0.6% of the rows at $h=60$ s, estimated from the mean rate). For the dropped rows $\Delta P^m$ would be zero, so the estimates at short $h$ are conditioned on activity; the effect on the sign statistic is of the order of the share of rows dropped, small against the width of the intervals.

In [ ]:
from unito26.lob.estimation import bootstrap_means, kendall_tau_b, politis_white_block_length
from unito26.lob.imbalance_regression import (
    forward_change, signed_covariance, signed_covariance_terms,
)
from unito26.lob.messages import Horizon

WINDOWS = (1.0, 5.0, 10.0, 30.0, 60.0)
HORIZONS = (1.0, 5.0, 10.0, 30.0, 60.0)
REPLICATES = 300
rng = np.random.default_rng(2026)
seconds_per_row = (mid.times[-1] - mid.times[0]) / len(mid)

results = []
for w in WINDOWS:
    x = backward_sum(flow, Window(w))
    for h in HORIZONS:
        dp = forward_change(mid, Horizon(h))
        ok = np.isfinite(x) & np.isfinite(dp)
        xs, ys, ts = x[ok], dp[ok], mid.times[ok]
        terms = signed_covariance_terms(xs, ys)  # columns: sign*change, sign, change
        cov = signed_covariance(xs, ys)
        block = max(politis_white_block_length(terms[:, 0]) * seconds_per_row, w + h)
        means = bootstrap_means(ts, terms, block, REPLICATES, rng)
        boot = means[:, 0] - means[:, 1] * means[:, 2]
        low, high = np.percentile(boot, [2.5, 97.5])
        results.append({
            "w (s)": w, "h (s)": h, "rows": int(ok.sum()), "block (s)": round(block, 1),
            "cov (ticks)": cov, "low": low, "high": high,
            "tau_b": kendall_tau_b(xs, ys),
            "read": "+" if low > 0 else ("-" if high < 0 else "."),
        })

grid_ofi = pd.DataFrame(results)
print(grid_ofi.round({"cov (ticks)": 4, "low": 4, "high": 4, "tau_b": 4}).to_string(index=False))

**Result.** The sign statistic is positive in every cell. The interval excludes zero in 20 cells out of 25:

- all horizons up to 60 s for $w\le 10$ s;
- $h\le 10$ s for $w=30$ s;
- $h\le 5$ s for $w=60$ s.

It includes zero at $(30,30)$, $(30,60)$, $(60,10)$, $(60,30)$ and $(60,60)$, where the sign cannot be read.

Kendall's $\tau_b$ is positive everywhere and small, from 0.167 at $(1,1)$ to 0.023 at $(60,60)$. At every $w$ it falls as $h$ grows, with one exception at $(60,30)$. Along $w$ it is not monotone: at $h=5$, $10$ and $30$ s it rises slightly from $w=1$ to $w=5$–$10$ s (0.134 to 0.139 at $h=5$ s) and falls from there on. The relation is therefore strongest at $(1,1)$ and weakest for $w=60$ s and long horizons.

## v. The same with $I^1_t$

### The grid for $I^1_t$ and the paired difference

We repeat the grid of point iv. with $I^1_t$ (1.1.2) in place of $\mathrm{OFI}_{t,w}$. $I^1_t$ is a state of the book and has no window, and it is never undefined, so for each $(w,h)$ the rows are those of point iv.: the two predictors are compared on one row set (docstring of `unito26.lob.imbalance_regression`).

The statistics are the same: $\mathrm{Cov}\big(\operatorname{sign} I^1_t,\Delta P^m\big)$, which depends only on the sign of the predictor and is therefore comparable with that of $\mathrm{OFI}$, and Kendall's $\tau_b$. The bootstrap resamples the terms of both predictors with the same blocks (`bootstrap_means` uses one index set for all columns), so the difference of the two sign statistics has a paired interval. The block duration is the largest of the two Politis-White lengths and $w+h$.

In [ ]:
tau_ofi = grid_ofi.set_index(["w (s)", "h (s)"])["tau_b"]

grid_rows, diff_rows = [], []
for w in WINDOWS:
    x = backward_sum(flow, Window(w))
    for h in HORIZONS:
        dp = forward_change(mid, Horizon(h))
        ok = np.isfinite(x) & np.isfinite(dp)
        ys, ts = dp[ok], mid.times[ok]
        terms_ofi = signed_covariance_terms(x[ok], ys)
        terms_i1 = signed_covariance_terms(i1[ok], ys)
        cov_ofi, cov_i1 = signed_covariance(x[ok], ys), signed_covariance(i1[ok], ys)
        block = max(max(politis_white_block_length(terms_ofi[:, 0]),
                        politis_white_block_length(terms_i1[:, 0])) * seconds_per_row, w + h)
        means = bootstrap_means(ts, np.column_stack([terms_ofi, terms_i1]), block, REPLICATES, rng)
        boot_ofi = means[:, 0] - means[:, 1] * means[:, 2]
        boot_i1 = means[:, 3] - means[:, 4] * means[:, 5]
        low, high = np.percentile(boot_i1, [2.5, 97.5])
        low_d, high_d = np.percentile(boot_ofi - boot_i1, [2.5, 97.5])
        tau_i1 = kendall_tau_b(i1[ok], ys)
        grid_rows.append({
            "w (s)": w, "h (s)": h, "rows": int(ok.sum()), "block (s)": round(block, 1),
            "cov (ticks)": cov_i1, "low": low, "high": high, "tau_b": tau_i1,
            "read": "+" if low > 0 else ("-" if high < 0 else "."),
        })
        diff_rows.append({
            "w (s)": w, "h (s)": h, "cov OFI": cov_ofi, "cov I1": cov_i1,
            "difference": cov_ofi - cov_i1, "low": low_d, "high": high_d,
            "tau OFI": tau_ofi[(w, h)], "tau I1": tau_i1,
            "read": "OFI" if low_d > 0 else ("I1" if high_d < 0 else "."),
        })

grid_i1, grid_diff = pd.DataFrame(grid_rows), pd.DataFrame(diff_rows)
print(grid_i1.round({"cov (ticks)": 4, "low": 4, "high": 4, "tau_b": 4}).to_string(index=False))

**Result.** $I^1_t$ does not depend on $w$, which only selects the rows, so at fixed $h$ the five cells are nearly the same estimate (at $h=1$ s the covariance is between 0.2554 and 0.2616).

The sign is positive up to $h=10$ s and the interval excludes zero there for every $w$: 15 cells out of 25. At $h=30$ s the estimate is still positive but the interval includes zero. At $h=60$ s the estimate is negative (between $-0.075$ and $-0.151$) and the interval includes zero, so no change of sign is read. Kendall's $\tau_b$ falls from 0.096 at $h=1$ s to 0.054 at 10 s and to about 0.005 at 30 s, and is slightly negative at 60 s.

### The comparison of the two predictors

For each $(w,h)$ the table sets the sign statistic of $\mathrm{OFI}_{t,w}$ against that of $I^1_t$ on the same rows, with Kendall's $\tau_b$ of both. The interval of the difference $\mathrm{OFI}-I^1$ is the paired one of the previous cell. The column `read` is `OFI` if the interval excludes zero and the difference is positive, `I1` if it is negative, `.` if the interval includes zero.

In [ ]:
print(grid_diff.round({"cov OFI": 4, "cov I1": 4, "difference": 4, "low": 4, "high": 4,
                       "tau OFI": 4, "tau I1": 4}).to_string(index=False))

**Result.** The estimate lies inside its interval in every cell. The difference of the sign statistics, $\mathrm{OFI}$ minus $I^1$, is positive in 24 cells out of 25 and its interval excludes zero in 16: all five horizons for $w\le10$ s, and $h=60$ s for $w=30$ s. For $w=60$ s no cell separates the two predictors, and the only negative difference, at $(60,1)$, has an interval that includes zero. No cell favours $I^1$.

Kendall's $\tau_b$ is larger for $\mathrm{OFI}$ in 22 cells out of 25. In the other three ($w=60$ s, $h\le10$ s) it is larger for $I^1$ (0.095 against 0.067 at $(60,1)$); there is no interval for $\tau_b$, so this is not read as a result. At $h=60$ s the difference is large because the sign statistic of $I^1$ is close to zero, not because $\mathrm{OFI}$ forecasts better there.

### What Remark 1.5.1 changes

In the generated setting the comparison of $I^1$ with $\mathrm{OFI}$ is read asymmetrically (Remark 1.5.1): the intensities do not read the book and there is no adverse selection, so whatever power $I^1$ keeps is mechanical, a depleted queue being one market order from clearing. On recorded data the flow is traded, a thin queue can be thin because those posting on that side inferred something (Section 1.5), and the comparison is symmetric. For AMZN on 21 June 2012, 10:00 to 13:00:

- **$\mathrm{OFI}$ against $I^1$.** $\mathrm{OFI}$ has the larger sign statistic for $w\le10$ s at every horizon up to 60 s, and at $h=60$ s for $w=30$ s; elsewhere the two are not separated. This is now a statement about this market, not about order-driven markets in general. No cell favours $I^1$, which in the generated setting would have been the stronger finding.
- **The sign of $I^1$.** It is positive and readable up to $h=10$ s. A mechanical and an informational effect would both give this, so the sign alone does not show that $I^1$ carries information in the sense of Section 1.5.
- **The sign of $\mathrm{OFI}$.** It is observed, not assumed: positive in every cell, with no cell of reversal, and readable in 20 of 25 (point iv.). $\theta$ of (1.4.13) can carry either sign (Corollary 1.4.21), but this does not identify it, since that needs Assumption 1.4.1.

For $\mathrm{OFI}$ the sign is readable for windows up to 10 s at all horizons up to 60 s, and for $I^1$ at horizons up to 10 s.